# Fase 2c: CT patch training sin fuga en test

Este notebook entrena CT con parches 128x128 sesgados hacia pixeles positivos de lesion durante entrenamiento. Validacion y test se hacen sobre slices completos 256x256, para no usar la mascara como ROI en evaluacion.

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.matplotlib_cache'))

import matplotlib.pyplot as plt
import pandas as pd

from src.config import config
from src.data.segmentation import build_ct_segmentation_dataframe, split_segmentation_dataframe
from src.training.segmentation_experiment import (
    existing_segmentation_artifact,
    get_device,
    make_segmentation_run_config,
    save_segmentation_artifacts,
    seed_everything,
    summarize_segmentation_results,
    train_and_evaluate_segmentation,
)

pd.set_option('display.max_columns', 50)
seed_everything(config.RANDOM_SEED)
device = get_device()
device

## 2. Configuracion experimental

El experimento se guarda como variante independiente y no pisa los baselines.

In [ ]:
run_config = make_segmentation_run_config(
    dataset_name='ct',
    architecture='attention_unet',
    run_mode='full',
    image_size=config.CT_IMAGE_SIZE,
    in_channels=1,
    variant_name='patch128_pos80_tversky_pos20_thr',
    loss_name='weighted_tversky_bce',
    bce_weight=0.2,
    dice_weight=0.8,
    pos_weight=20.0,
    tversky_alpha=0.3,
    tversky_beta=0.7,
    optimize_threshold=True,
    train_crop_size=(128, 128),
    positive_crop_prob=0.8,
    base_features=16,
    batch_size=16,
    epochs=15,
    early_stopping_patience=5,
    learning_rate=1e-4,
)

seg_models_dir = config.MODELS_DIR / 'segmentation' / 'ct'
seg_results_dir = PROJECT_ROOT / 'results' / 'segmentation' / 'ct'
processed_seg_dir = config.CT_DIR / 'processed_segmentation_slices'
seg_models_dir.mkdir(parents=True, exist_ok=True)
seg_results_dir.mkdir(parents=True, exist_ok=True)

print(f'Experimento: {run_config.experiment_name}_{run_config.run_mode}')
print(f'Train crop: {run_config.train_crop_size}, positive_crop_prob={run_config.positive_crop_prob}')
print(f'Evaluacion: slices completos {run_config.image_size}')

## 3. Datos y splits

In [ ]:
ct_seg_df = build_ct_segmentation_dataframe(
    ct_dir=config.CT_DIR,
    output_dir=processed_seg_dir,
    target_size=config.CT_IMAGE_SIZE,
    positive_mask_only=True,
    overwrite=False,
)
train_df, val_df, test_df = split_segmentation_dataframe(
    ct_seg_df,
    random_seed=config.RANDOM_SEED,
    group_col='study_id',
)

display(pd.DataFrame([
    {'split': 'train', 'slices': len(train_df), 'studies': train_df['study_id'].nunique()},
    {'split': 'val', 'slices': len(val_df), 'studies': val_df['study_id'].nunique()},
    {'split': 'test', 'slices': len(test_df), 'studies': test_df['study_id'].nunique()},
]))

## 4. Entrenamiento

In [ ]:
if existing_segmentation_artifact(run_config, seg_models_dir, seg_results_dir):
    print('Saltado: esta variante ya tiene modelo y summary guardados.')
    latest_result = None
else:
    latest_result = train_and_evaluate_segmentation(
        run_config=run_config,
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        device=device,
    )
    saved_paths = save_segmentation_artifacts(run_config, latest_result, seg_models_dir, seg_results_dir)
    print(saved_paths)

## 5. Comparacion CT

In [ ]:
summary_df = summarize_segmentation_results(sorted(seg_results_dir.glob('*_summary.json')))
cols = ['experiment', 'variant_name', 'loss_name', 'dice', 'iou', 'pixel_accuracy', 'threshold', 'hyperparameters']
display(summary_df.sort_values('dice', ascending=False)[[c for c in cols if c in summary_df.columns]])

## 6. Visualizacion cualitativa

In [ ]:
if latest_result is not None:
    examples = latest_result['examples'][:4]
    threshold = latest_result.get('threshold', 0.5)
    fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
    if len(examples) == 1:
        axes = axes[None, :]
    for row_idx, example in enumerate(examples):
        image = example['image'].squeeze().numpy()
        mask = example['mask'].squeeze().numpy()
        pred = (example['prediction'].squeeze().numpy() >= threshold).astype(float)
        axes[row_idx, 0].imshow(image, cmap='gray')
        axes[row_idx, 0].set_title('Imagen completa')
        axes[row_idx, 1].imshow(mask, cmap='gray')
        axes[row_idx, 1].set_title('Mascara real')
        axes[row_idx, 2].imshow(pred, cmap='gray')
        axes[row_idx, 2].set_title(f'Prediccion t={threshold:.2f}')
        for ax in axes[row_idx]:
            ax.axis('off')
    plt.tight_layout()
else:
    print('No hay entrenamiento nuevo en esta ejecucion.')